# Encoding Model for Brain Responses

This notebook implements an encoding model to predict brain responses based on various inputs, incorporating confidence scores related to memory. It analyzes whether higher confidence correlates with increased activity in specific brain regions and includes functionality to plot brain maps, coloring the brain based on correlations or prediction accuracy.

In [ ]:
# %% [markdown]
# Encoding model for brain responses with memory confidence and brain maps

# %% [code]
import sys
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from new_code import load_data_into_dict, slicer, label, feature

sns.set(style="whitegrid", font_scale=1.1)
warnings.filterwarnings("ignore", category=FutureWarning)

# %% [code]
NWB_DIR = Path("C:/Users/cierr/OneDrive/Documents/bmovie/nwb_files")
BIDS_DIR = Path("C:/Users/cierr/OneDrive/Documents/bmovie/bids_files")
ANNOT_PATH = repo_root / "assets" / "annotations" / "short_faceannots.pkl"
assert ANNOT_PATH.exists(), f"Annotation file not found: {ANNOT_PATH}"

sub_id = 41

# %% [code]
with open(ANNOT_PATH, "rb") as f:
    face_annots = pickle.load(f)

df_face_note = label.build_face_feature_df(face_annots, target_id="Jacky")

print("Face annotation columns:", df_face_note.columns.tolist()[:40])
print("Face annotation rows:", len(df_face_note))
df_face_note.head()

# %% [code]
data = load_data_into_dict.load_multimodal_subjects(
    sub_nums=[sub_id],
    nwb_root=NWB_DIR,
    bids_root=BIDS_DIR,
    max_nwb_samples=100_000,
    load_fmri=False,
    resample_lfp=False,
    verbose=False,
)
sub_data = data[sub_id]

print("Available modalities:", [k for k in sub_data.keys() if k not in ("time_grid", "movie_time", "meta")])
print("time_grid length:", len(sub_data["time_grid"]))

# %% [code]
def detect_confidence_columns(df):
    return [c for c in df.columns if "confidence" in c.lower() or "conf" in c.lower()]

confidence_cols = detect_confidence_columns(df_face_note)
print("Detected confidence columns:", confidence_cols)

# %% [code]
face_feature_sets = {
    "quantity": ["n_faces", "total_face_area", "max_face_area"],
    "identity": ["Jacky_present"],
    "pose_emotion": [
        "any_front_face",
        "emo_afraid", "emo_angry", "emo_happy",
        "emo_neutral", "emo_surprised",
    ],
    "all_face": [
        "n_faces", "total_face_area", "max_face_area",
        "Jacky_present", "any_front_face",
        "emo_afraid", "emo_angry", "emo_happy",
        "emo_neutral", "emo_surprised",
    ],
}

if confidence_cols:
    face_feature_sets["memory_confidence"] = confidence_cols
    face_feature_sets["face_all_plus_confidence"] = face_feature_sets["all_face"] + confidence_cols

method_map = {
    "n_faces": "nearest",
    "total_face_area": "linear",
    "max_face_area": "linear",
    "Jacky_present": "nearest",
    "any_front_face": "nearest",
    "emo_afraid": "nearest",
    "emo_angry": "nearest",
    "emo_happy": "nearest",
    "emo_neutral": "nearest",
    "emo_surprised": "nearest",
}
for col in confidence_cols:
    method_map[col] = "nearest"

lags_sec = (-0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.5)

# %% [code]
def build_feature_matrix(df_face, sub_dict, cols, lags_sec, method_map):
    t_grid = np.asarray(sub_dict["time_grid"], dtype=float)
    dt = float(np.median(np.diff(t_grid)))
    lags = np.round(np.asarray(lags_sec) / dt).astype(int)

    df_face_grid = slicer.resample_face_df_to_timegrid(
        df_face, t_grid, cols, method_map=method_map
    )

    X = df_face_grid[cols].astype(float).to_numpy()
    X_lag = feature.make_lagged_matrix(X, lags)
    return X_lag

def flatten_targets(Y, target_prefix):
    Y = np.asarray(Y, dtype=float)
    if Y.ndim == 1:
        return Y[:, None], [f"{target_prefix}_0"]
    elif Y.ndim == 2:
        names = [f"{target_prefix}_{i}" for i in range(Y.shape[1])]
        return Y, names
    elif Y.ndim == 3:
        T, A, B = Y.shape
        Y2d = Y.reshape(T, A * B)
        names = [f"{target_prefix}_{a}_{b}" for a in range(A) for b in range(B)]
        return Y2d, names
    else:
        raise ValueError(f"Unsupported Y.ndim={Y.ndim}")

def ridge_encoding_cv_single_output(X, y, alpha=10.0, n_splits=5):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)

    if X.ndim == 1:
        X = X[:, None]
    if len(X) != len(y):
        raise ValueError(f"X and y length mismatch: {len(X)} vs {len(y)}")

    valid = np.all(np.isfinite(X), axis=1) & np.isfinite(y)
    X = X[valid]
    y = y[valid]

    T = len(y)
    if T < n_splits:
        return {"corr": np.nan, "r2": np.nan, "pred": np.full(T, np.nan)}

    fold_edges = np.linspace(0, T, n_splits + 1).astype(int)
    preds = np.full(T, np.nan, dtype=float)

    for i in range(n_splits):
        te0, te1 = fold_edges[i], fold_edges[i + 1]
        te_idx = np.arange(te0, te1)
        tr_idx = np.setdiff1d(np.arange(T), te_idx)

        xtr, xte = X[tr_idx], X[te_idx]
        ytr = y[tr_idx]

        x_scaler = StandardScaler()
        xtr = x_scaler.fit_transform(xtr)
        xte = x_scaler.transform(xte)

        model = Ridge(alpha=alpha)
        model.fit(xtr, ytr)
        preds[te_idx] = model.predict(xte)

    valid_pred = np.isfinite(preds)
    if valid_pred.sum() < 3:
        return {"corr": np.nan, "r2": np.nan, "pred": preds}

    yv = y[valid_pred]
    pv = preds[valid_pred]
    corr = np.corrcoef(yv, pv)[0, 1] if np.std(yv) > 0 and np.std(pv) > 0 else np.nan
    r2 = 1.0 - np.sum((yv - pv) ** 2) / np.sum((yv - np.mean(yv)) ** 2) if np.var(yv) > 0 else np.nan

    return {"corr": corr, "r2": r2, "pred": preds}

def run_encoding_for_modality(X, Y, modality_name, alpha=10.0, n_splits=5, max_targets=None):
    if isinstance(Y, dict):
        dfs = []
        for band, arr in Y.items():
            df_band = run_encoding_for_modality(
                X,
                arr,
                f"{modality_name}_{band}",
                alpha=alpha,
                n_splits=n_splits,
                max_targets=max_targets,
            )
            dfs.append(df_band)
        return pd.concat(dfs, axis=0, ignore_index=True)

    Y2d, target_names = flatten_targets(Y, modality_name)
    if max_targets is not None:
        Y2d = Y2d[:, :max_targets]
        target_names = target_names[:max_targets]

    rows = []
    for j, target_name in enumerate(target_names):
        y = Y2d[:, j]
        res = ridge_encoding_cv_single_output(X, y, alpha=alpha, n_splits=n_splits)
        rows.append({
            "modality": modality_name,
            "target": target_name,
            "corr": res["corr"],
            "r2": res["r2"],
        })
    return pd.DataFrame(rows)

def run_face_encoding_all_modalities(
    df_face_note,
    sub_dict,
    face_feature_sets,
    method_map,
    lags_sec,
    alpha=10.0,
    n_splits=5,
    max_targets_per_modality=None,
):
    results = []
    for model_name, cols in face_feature_sets.items():
        X = build_feature_matrix(df_face_note, sub_dict, cols, lags_sec, method_map)
        modality_specs = {
            "firing_rate": sub_dict.get("firing_rate", None),
            "lfp_macro": sub_dict.get("lfp_macro", None),
            "lfp_micro": sub_dict.get("lfp_micro", None),
            "lfp_bandpower": sub_dict.get("lfp_bandpower", None),
            "pupil": sub_dict.get("pupil", None),
            "eye_gaze": sub_dict.get("eye_gaze", None),
        }
        for modality_name, Y in modality_specs.items():
            if Y is None:
                continue
            try:
                df_res = run_encoding_for_modality(
                    X=X,
                    Y=Y,
                    modality_name=modality_name,
                    alpha=alpha,
                    n_splits=n_splits,
                    max_targets=max_targets_per_modality,
                )
                df_res["model_name"] = model_name
                results.append(df_res)
            except Exception as exc:
                results.append(pd.DataFrame([{
                    "modality": modality_name,
                    "target": None,
                    "corr": np.nan,
                    "r2": np.nan,
                    "model_name": model_name,
                    "error": repr(exc),
                }]))
    if len(results) == 0:
        return pd.DataFrame()
    return pd.concat(results, axis=0, ignore_index=True)

# %% [code]
df_encoding = run_face_encoding_all_modalities(
    df_face_note=df_face_note,
    sub_dict=sub_data,
    face_feature_sets=face_feature_sets,
    method_map=method_map,
    lags_sec=lags_sec,
    alpha=10.0,
    n_splits=5,
    max_targets_per_modality=20,
)

df_encoding.head()

# %% [code]
summary = df_encoding.groupby(["modality", "model_name"])[["corr", "r2"]].mean().reset_index()
summary

# %% [code]
plt.figure(figsize=(12, 5))
sns.barplot(
    data=summary[summary["modality"].isin(["firing_rate", "lfp_macro", "lfp_micro"])],
    x="modality", y="corr", hue="model_name",
)
plt.title("Mean encoding correlation by modality and feature set")
plt.xticks(rotation=25)
plt.tight_layout()

# %% [code]
from pynwb import NWBHDF5IO

def load_unit_locations(nwb_root, subject_id):
    nwb_sub = load_data_into_dict.nwb_subject_from_int(subject_id)
    nwb_path = load_data_into_dict.find_nwb_file_for_subject(nwb_root, nwb_sub)
    with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
        nwb = io.read()
        if getattr(nwb, "units", None) is None:
            return None
        unit_df = nwb.units.to_dataframe()
        if {"x", "y", "z"}.issubset(unit_df.columns):
            return unit_df[["x", "y", "z"]].reset_index(drop=True)

        if "ecephys_electrodes" not in unit_df.columns:
            return None

        electrode_df = nwb.electrodes.to_dataframe()
        coords = []
        for ref in unit_df["ecephys_electrodes"]:
            if ref is None or len(ref) == 0:
                coords.append((np.nan, np.nan, np.nan))
                continue
            first_ref = ref[0]
            if hasattr(first_ref, "index"):
                idx = int(first_ref.index)
            else:
                idx = int(first_ref)
            coords.append((electrode_df.loc[idx, "x"], electrode_df.loc[idx, "y"], electrode_df.loc[idx, "z"]))
        coords_df = pd.DataFrame(coords, columns=["x", "y", "z"])
        return coords_df

coords_df = load_unit_locations(NWB_DIR, sub_id)
print("Unit coordinates loaded:", coords_df is not None)

# %% [code]
def plot_unit_metric(coords, values, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    sc = ax.scatter(coords["x"], coords["y"], c=values, cmap="coolwarm", s=80, edgecolor="k")
    plt.colorbar(sc, ax=ax, label="metric")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.set_aspect("equal", "box")
    plt.tight_layout()
    plt.show()

if coords_df is not None:
    fr_summary = df_encoding[df_encoding["modality"] == "firing_rate"].copy()
    fr_summary["unit_id"] = fr_summary["target"].str.extract(r"_(\d+)$").astype(float)
    fr_summary = fr_summary.dropna(subset=["unit_id"])
    fr_summary["unit_id"] = fr_summary["unit_id"].astype(int)
    avg_unit = fr_summary.groupby("unit_id")[["corr", "r2"]].mean().reset_index()
    merged = avg_unit.merge(coords_df.reset_index().rename(columns={"index": "unit_id"}), on="unit_id", how="inner")
    if not merged.empty:
        plot_unit_metric(merged, merged["corr"], "Firing rate encoding correlation per unit")
        plot_unit_metric(merged, merged["r2"], "Firing rate encoding R² per unit")
    else:
        print("No matching unit coordinates found for the encoding results.")
else:
    print("No electrode/unit coordinates available in this NWB file.")